# A2 — Knowledge-Base Demo

This notebook demonstrates the implemented A2 knowledge base on the current built artefacts. It reports an OCR pipeline-quality metric from the OCR manifest, computes ground-truth CER/WER when the held-out labels are populated, and performs one real FAISS retrieval using the project's BGE-M3 embedder.

**A1 profile:** high-school STEM mathematics; data speciality = math/scientific notation; primary NFR = Explainable. The notebook does not fabricate a CER/WER value when `grading_kit/labels.jsonl` still contains the starter placeholder.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np

from doc_agent.config import load as load_config
from doc_agent.contracts import Chunk
from doc_agent.index.embed import encode
from doc_agent.index.store import load as load_store

ROOT = Path.cwd()
CFG = load_config()
OCR_MANIFEST = ROOT / CFG.get('ocr', {}).get('manifest_path', 'data/interim/ocr/chunks.jsonl')
CHUNK_MANIFEST = ROOT / CFG.get('chunk', {}).get('manifest_path', 'data/interim/chunks/chunks.jsonl')
LABELS_PATH = ROOT / 'grading_kit' / 'labels.jsonl'

print('Project root:', ROOT)
print('OCR manifest:', OCR_MANIFEST)
print('Chunk manifest:', CHUNK_MANIFEST)
print('Labels:', LABELS_PATH)

## 1. OCR pipeline-quality diagnostic

The first metric is **OCR usable-region rate**: successful non-skipped OCR records divided by all non-skipped records. This is a pipeline-health metric, not a substitute for ground-truth CER/WER. It is useful for showing how many layout regions survive OCR and quality filtering.

In [ ]:
rows = [json.loads(line) for line in OCR_MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()]
non_skipped = [row for row in rows if row.get('status') != 'skipped']
ok_rows = [row for row in non_skipped if row.get('status') == 'ok' and str(row.get('text', '')).strip()]
rejected_rows = [row for row in non_skipped if row.get('status') != 'ok']

usable_rate = len(ok_rows) / len(non_skipped) if non_skipped else float('nan')
print(f'Total OCR records: {len(rows)}')
print(f'Non-skipped records: {len(non_skipped)}')
print(f'Usable OCR records: {len(ok_rows)}')
print(f'Rejected non-skipped records: {len(rejected_rows)}')
print(f'OCR usable-region rate: {usable_rate:.4%}')

## 2. Ground-truth OCR quality: CER/WER when labels are available

A1/A2 requires a set-aside holdout with exact transcriptions. This cell computes character error rate (CER) and word error rate (WER) from `grading_kit/labels.jsonl` without using the PDF text layer as an OCR shortcut. If the file still contains the starter `REPLACE ME` placeholder, the notebook reports that the oracle is not ready instead of inventing a score.

In [ ]:
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        cur = [i]
        for j, y in enumerate(b, start=1):
            cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def cer(reference, hypothesis):
    reference = reference.strip()
    return edit_distance(list(reference), list(hypothesis)) / max(1, len(reference))

def wer(reference, hypothesis):
    ref_words = re.findall(r'\S+', reference.strip())
    hyp_words = re.findall(r'\S+', hypothesis.strip())
    return edit_distance(ref_words, hyp_words) / max(1, len(ref_words))

labels = [json.loads(line) for line in LABELS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
valid_labels = [row for row in labels if str(row.get('text', '')).strip() and 'REPLACE ME' not in str(row.get('text', ''))]

if not valid_labels:
    print('Ground-truth CER/WER: NOT AVAILABLE')
    print('Reason: grading_kit/labels.jsonl does not yet contain real held-out transcriptions.')
    print('Do not report a fabricated OCR accuracy number; populate the holdout oracle before final A2 submission.')
else:
    by_page = {}
    for row in rows:
        if row.get('status') == 'ok' and str(row.get('text', '')).strip():
            by_page.setdefault(str(row['page_id']), []).append(row)

    total_chars = 0
    total_words = 0
    char_errors = 0
    word_errors = 0
    evaluated = 0

    for label in valid_labels:
        page_id = str(label['page_id'])
        prediction_rows = sorted(by_page.get(page_id, []), key=lambda r: int(r.get('order', 0)))
        if not prediction_rows:
            continue
        hypothesis = '\n'.join(str(r['text']) for r in prediction_rows).strip()
        reference = str(label['text']).strip()
        char_errors += edit_distance(list(reference), list(hypothesis))
        word_errors += edit_distance(re.findall(r'\S+', reference), re.findall(r'\S+', hypothesis))
        total_chars += len(reference)
        total_words += len(re.findall(r'\S+', reference))
        evaluated += 1

    print(f'Holdout pages evaluated: {evaluated}/{len(valid_labels)}')
    print(f'CER: {char_errors / max(1, total_chars):.4%}')
    print(f'WER: {word_errors / max(1, total_words):.4%}')

## 3. One real retrieval from the persistent FAISS index

The query below is built from the first persisted chunk so the demo remains runnable on different smoke corpora. It is a genuine BGE-M3 embedding followed by FAISS inner-product search; the result includes chunk and page provenance. This is a retrieval smoke test, not the final A3 retrieval evaluation.

In [ ]:
index, records = load_store(CFG)
assert index.ntotal == len(records), f'Index/metadata mismatch: {index.ntotal} vs {len(records)}'
assert index.ntotal > 0, 'The FAISS index is empty; run scripts/run_ingest.py first.'

source_record = records[0]
source_text = str(source_record['text']).strip()
query = ' '.join(source_text.split()[:18])
query_chunk = Chunk(
    id='__demo_query__',
    doc_id='__demo__',
    text=query,
    page_ids=[],
)
query_vector = encode([query_chunk], CFG).astype(np.float32)
scores, indices = index.search(query_vector, 3)

print('Query:', query)
print('\nTop-3 retrieved chunks:')
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    record = records[int(idx)]
    print(f'#{rank} score={float(score):.4f} chunk_id={record["chunk_id"]} pages={record["page_ids"]}')
    print('   ', record['text'][:240].replace('\n', ' '))

top1 = records[int(indices[0][0])]
print('\nTop-1 self-retrieval hit:', top1['chunk_id'] == source_record['chunk_id'])


## A2 evidence summary

The notebook demonstrates the two required knowledge-base checks without fabricating evidence: (1) an OCR pipeline-quality metric plus ground-truth CER/WER when the real holdout oracle exists, and (2) one real retrieval from the persisted FAISS index with page provenance. For the final A2 submission, replace the placeholder holdout labels with the actual held-out page transcriptions and rerun this notebook so the CER/WER cells produce the real reported numbers.